# eXplainable AI (SHAP) for Keras Model
We use `shap.DeepExplainer` to interpret the deep learning model predictions.


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
shap.initjs()


In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
if 'id' in df.columns: df.drop('id', axis=1, inplace=True)
df = df[df['gender'] != 'Other']
df_impute = df.copy()
for col in df_impute.select_dtypes(include=['object', 'string']).columns:
    df_impute[col] = LabelEncoder().fit_transform(df_impute[col].astype(str))
df['bmi'] = KNNImputer(n_neighbors=2).fit_transform(df_impute)[:, df_impute.columns.get_loc('bmi')].round(1)
df = df[(df['bmi'] >= df['bmi'].quantile(0.001)) & (df['bmi'] <= df['bmi'].quantile(0.999))]
df['work_type'] = df['work_type'].map({'Govt_job': 4, 'Never_worked': 1, 'Private': 3, 'Self-employed': 2, 'children': 0})
df['smoking_status'] = df['smoking_status'].map({'Unknown': 1, 'formerly smoked': 2, 'never smoked': 0, 'smokes': 3})
for col in ['gender', 'ever_married', 'Residence_type']: df[col] = LabelEncoder().fit_transform(df[col])
X = df.drop('stroke', axis=1)
y = df['stroke'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=59)
X_res, y_res = smote.fit_resample(X_scaled, y)
model = Sequential([Dense(128, activation='relu', input_dim=X_res.shape[1]), Dense(64, activation='relu'), Dense(32, activation='relu'), Dense(1, activation='sigmoid')])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_res, y_res, epochs=20, batch_size=32, verbose=0)


## Explaining Global Feature Importance with SHAP


In [ ]:
tf.compat.v1.disable_v2_behavior() # Sometimes required for DeepExplainer in TF2
background = X_scaled[np.random.choice(X_scaled.shape[0], 100, replace=False)]
explainer = shap.DeepExplainer(model, background)

shap_values = explainer.shap_values(X_scaled[:200])

if isinstance(shap_values, list):
    shap_vals_positive = shap_values[0]
else:
    shap_vals_positive = shap_values


In [ ]:
shap.summary_plot(shap_vals_positive, X_scaled[:200], feature_names=X.columns)


## Local Explanation for a Single Patient


In [ ]:
sample_idx = 0
shap.force_plot(explainer.expected_value[0], shap_vals_positive[sample_idx,:], X_scaled[sample_idx,:], feature_names=X.columns)
